<a href="https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item, for one client, aggregated over one month.**

The daily fact table's grain is `report_date + client_hash_id + content_hash_id`
— one row per page per client per day. That is not my decision grain. A reviewer
does not act on a page-day, they act on a page. So I collapse the daily rows over
my window down to one row per `client_hash_id + content_hash_id`, and that
aggregated row is my unit of analysis. Query 1 below verifies both the daily
grain and what the collapse leaves behind.

**The window is `month = 2026-03`, all 31 days.**

Two reasons for a mid-panel month. The panel is unbalanced — clients start
tracking at different dates — so a middle month has more clients with real
history than either end. And the final month, June 2026, is the natural outcome
window for any past-to-future label, so I am treating it as a sealed test month
and will not develop against it. That also rules out the `_sample` table, which
is exactly that final month.

**What I predict or rank.** The regression target is observed CTR over the
window — `SUM(clicks) / SUM(impressions)` from the daily rows. From it I derive
the ranking quantity: the residual between a page's actual CTR and the CTR a
model expects for a page with its position, exposure, type, intent, and age. I am
calling that residual a **provisional proxy**, not an outcome: it says a page
looks unusual now, not that the page later improved. The future-window label
comes once I have a past-to-future window pair defined.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features — five, all aggregated to my decision grain over the window:**

| Field | Source | Bucket |
|---|---|---|
| `avg_position` | `SUM(gsc_sum_position) / SUM(gsc_impressions)` from the daily table | feature |
| `impressions` | `SUM(gsc_impressions)` from the daily table | feature |
| `content_type` | `dim_content` | feature |
| `main_intent` | `dim_content` | feature |
| `content_age_days` | `dim_content.content_created_date` measured to the window start | feature |

Position is impression-weighted rather than a plain mean of the daily averages,
because a 5-impression day and a 5,000-impression day should not carry the same
weight. The schema ships `gsc_sum_position` precisely so this is possible.

**Label / target:** observed CTR, `SUM(gsc_clicks) / SUM(gsc_impressions)` over
the window. It is measured in the data, not defined by a rule.

**Context — used, but never as features:**

- `client_hash_id` — for grouped validation only. One client can dominate a
  slice, so a random split would let the same client sit on both sides.
- `content_hash_id` — join key between the daily facts and `dim_content`.
- `gsc_data_available` — an availability check, not a signal. Query 3 shows why
  it turns out to carry no information beyond "this row has impressions".

**Excluded, and why:**

- **`gsc_clicks` as a standalone feature.** CTR is clicks over impressions, and
  impressions is already a feature — so clicks is half of my own label. This is
  the trap I demonstrate deliberately in section 3 rather than argue about.
- **`fact_content_query_90d`.** Its query-mix features are genuinely tempting,
  but they are computed over a fixed 90-day window I do not control, and that
  window will overlap whatever target window I define. I would rather lose the
  features than lose the ability to trust the result.
- **`sessions_ai` and the per-platform AI columns.** Only 30,177 rows across the
  full 78.8M-row table carry AI sessions. Too sparse to be a stable feature at
  my grain.
- **GA4 engagement columns** (`ga4_sessions`, `scroll_events`, and the rest).
  They describe what happened after the click, and my question is about the
  click itself. Keeping the field list tight is deliberate — five features I can
  defend beat fifteen I cannot.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 — grain.** 9,841,378 daily rows and 9,841,378 distinct
`report_date + client_hash_id + content_hash_id` keys. The daily grain is exactly
what my contract claims. Collapsing to my decision grain leaves 331,437 unique
client-page pairs.

**Query 2 — slice size and span.** March 2026 holds 9,841,378 rows across all 31
days for 55 clients. Note that 331,437 pages over 31 days would be 10.27M rows if
every page appeared every day — I have 9.84M, so the panel is slightly ragged and
a page's monthly aggregate is a sum over the days it actually appears, not a
fixed 31.

**Query 3 — availability, checked with `IS TRUE`.** Of 9,841,378 rows, 3,611,061
have `gsc_data_available IS TRUE`, and exactly 3,611,061 have
`gsc_impressions > 0`. The two counts are identical.

That identity is the finding. The availability flag carries no information beyond
"this row has impressions", so **I cannot separate a page that was tracked and
served zero impressions from a page that was not tracked at all**. Only 37% of
daily rows carry usable search data, and for the missing 63% I know nothing about
why. Any statement I make about pages with low exposure has to be read with that
in mind.

**The five features, and why each is knowable at the decision moment.**

| Feature | Knowable because… |
|---|---|
| `avg_position` | impression-weighted from `SUM(gsc_sum_position) / SUM(gsc_impressions)` over the window — a measurement of where the page already ranked. A plain mean of the daily averages would weight a 5-impression day the same as a 5,000-impression day, so I used the weighted form the schema provides |
| `impressions` | summed over the window; it counts times the page was already shown. Exposure, not response |
| `content_type` | page metadata, fixed before the window opened |
| `main_intent` | same — a property of the page, not an outcome |
| `content_age_days` | measured against the window start (2026-03-01), never against today |

**The deliberate leak.** With the five honest features and client-grouped
validation: **R² 0.012, MAE 0.3101**. Adding `clicks` as a sixth feature:
**R² 0.993, MAE 0.0107**.

The jump is not skill. My label is `clicks / impressions`, and `impressions` was
already a feature — so handing the model `clicks` gives it both halves of the
answer. It is not predicting, it is dividing. The column is dropped and the
honest number is the one I keep.

**What the honest number tells me.** R² of 0.012 is close to zero: these five
features barely explain CTR at all. The distribution is a likely reason — over a
quarter of pages have exactly zero clicks despite 100+ impressions, while the top
of the range reaches 13.5%. A single regression on the raw rate is probably the
wrong shape for that. My expected-CTR approach depends on being able to predict
expected CTR, so this is the first thing I have to address in the modelling
weeks, not something to paper over.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
from google.colab import userdata
from huggingface_hub import login, list_repo_files
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
print(f"{len(files)} files total\n")
for f in sorted(files)[:40]:
    print(f)

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

TABLES = {
    "daily":   f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet')",
    "content": f"read_parquet('{BASE}/dim_content.parquet')",
    "clients": f"read_parquet('{BASE}/dim_clients.parquet')",
}

print("=== daily fact columns ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['daily']}").df().to_string())

print("\n=== dim_content columns ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['content']}").df().to_string())

# Query 1: grain
q1 = con.sql(f"""
    SELECT
        COUNT(*) AS daily_rows,
        COUNT(DISTINCT concat_ws('|', report_date::VARCHAR, client_hash_id, content_hash_id)) AS distinct_daily_keys,
        COUNT(DISTINCT concat_ws('|', client_hash_id, content_hash_id)) AS distinct_page_keys
    FROM {TABLES['daily']}
""").df()
print("Q1 — grain check")
print(q1.to_string(index=False))

# Query 2: row count + date span
q2 = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS first_day,
        MAX(report_date) AS last_day,
        COUNT(DISTINCT report_date) AS days_covered,
        COUNT(DISTINCT client_hash_id) AS clients
    FROM {TABLES['daily']}
""").df()
print("\nQ2 — slice size and date span")
print(q2.to_string(index=False))

#  Query 3: availability, with IS TRUE
q3 = con.sql(f"""
    SELECT
        COUNT(*) AS all_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND gsc_impressions > 0) AS rows_with_impressions
    FROM {TABLES['daily']}
""").df()
print("\nQ3 — availability (IS TRUE)")
print(q3.to_string(index=False))

# Five features, aggregated to my decision grain
frame = con.sql(f"""
    WITH agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions)                                  AS impressions,
            SUM(gsc_clicks)                                       AS clicks,
            SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)    AS avg_position,
            COUNT(*)                                              AS days_present
        FROM {TABLES['daily']}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        a.client_hash_id,
        a.content_hash_id,
        a.avg_position,
        a.impressions,
        c.content_type,
        c.main_intent,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-01') AS content_age_days,
        a.clicks,
        a.clicks * 100.0 / a.impressions AS ctr
    FROM agg a
    JOIN {TABLES['content']} c USING (client_hash_id, content_hash_id)
    WHERE a.impressions >= 100
      AND a.avg_position BETWEEN 1 AND 20
""").df()

print(f"Feature frame: {len(frame):,} rows, one per client-page")
print(frame[["avg_position", "impressions", "content_age_days", "ctr"]].describe().to_string())

base = frame[["avg_position", "impressions", "content_age_days", "content_type", "main_intent"]]
X_honest = pd.get_dummies(base, columns=["content_type", "main_intent"], dummy_na=True)
y = frame["ctr"]
groups = frame["client_hash_id"]

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
              .split(X_honest, y, groups))

def quick_score(X, label):
    m = HistGradientBoostingRegressor(random_state=42).fit(X.iloc[tr], y.iloc[tr])
    p = m.predict(X.iloc[te])
    print(f"{label:32s} R2 = {r2_score(y.iloc[te], p):6.3f}   MAE = {mean_absolute_error(y.iloc[te], p):.4f}")

quick_score(X_honest, "Honest features (5)")

# --- the deliberate leak: clicks is the numerator of my own label ---
X_leaky = X_honest.copy()
X_leaky["clicks"] = frame["clicks"]
quick_score(X_leaky, "With clicks added (leaky)")

del X_leaky
print("\nLeaky column dropped. The honest number above is the one I keep.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


24 files total

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Q1 — grain check
 daily_rows  distinct_daily_keys  distinct_page_keys
    9841378              9841378              331437

Q2 — slice size and date span
   rows  first_day   last_day  days_covered  clients
9841378 2026-03-01 2026-03-31            31       55


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Q3 — availability (IS TRUE)
 all_rows  gsc_available_rows  rows_with_impressions
  9841378             3611061                3611061


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 76,845 rows, one per client-page
       avg_position    impressions  content_age_days           ctr
count  76845.000000   76845.000000      76845.000000  76845.000000
mean       7.627189    2830.362847        153.523665      0.304847
std        4.574659    7023.339605        127.155534      0.445815
min        1.000000     100.000000        -29.000000      0.000000
25%        4.148973     337.000000         39.000000      0.000000
50%        6.360759     913.000000        136.000000      0.175234
75%       10.108280    2693.000000        233.000000      0.428266
max       20.000000  617124.000000        464.000000     13.526570
Honest features (5)              R2 =  0.012   MAE = 0.3101
With clicks added (leaky)        R2 =  0.993   MAE = 0.0107

Leaky column dropped. The honest number above is the one I keep.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**The availability flag carries no extra information.** Of 9,841,378 rows in
March 2026, exactly 3,611,061 have `gsc_data_available IS TRUE` and exactly
3,611,061 have `gsc_impressions > 0` — identical counts. So I cannot separate a
page that was tracked and served zero impressions from a page that was not
tracked at all. For the other 63% of rows I know nothing about why the data is
absent. This is the named limitation of my slice: any claim I make about
low-exposure pages is a claim about the 37% I can see.

**The panel is ragged.** 331,437 pages across 31 days would be 10.27M rows if
every page appeared daily; I have 9.84M. A monthly aggregate is therefore a sum
over the days a page actually appears, not a fixed 31, and pages with fewer days
carry noisier averages.

**Age can be negative.** `content_age_days` reaches -29 in my frame — pages
created after my window opened. Those pages did not exist at the decision moment
I am modelling, so they should be filtered out rather than fed in with a negative
age. Found here, fixed before modelling.

**What this data can never tell me.** Whether editing a page caused clicks to
change. I observe impressions, clicks, and position after the fact; I never see
the ranking system, the results page, or the user. Even a perfect model here
produces a decision-support ranking — pages worth a reviewer's time — and never a
causal claim.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.